# 03 - Train VAE on processed expression data

This notebook:

1. Loads processed training, validation, and test data
2. Defines a Variational Autoencoder (VAE)
3. Trains the model
4. Extracts latent representations from the full scaled dataset
5. Saves latent-space and model artifacts for downstream analysis

In [1]:
%run ./00_latent_space_config.ipynb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Notebook directory : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\projects\cancer-latent-space\notebooks
Project directory  : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity
ML input directory : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs
Processed directory: C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed
Output directory   : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\output\global_cancer
Expression path: C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\ml_inputs\hu35ksuba_expr_top3000_variance.csv | exists: True
Metadata path  : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\p

In [2]:
print("PROCESSED_DIR       :", PROCESSED_DIR)
print("TRAINING_PLOTS_DIR :", TRAINING_PLOTS_DIR)
print("LATENT_PLOTS_DIR   :", LATENT_PLOTS_DIR)
print("MODELS_DIR         :", MODELS_DIR)

for required_path in [X_TRAIN_PATH, X_VAL_PATH, X_TEST_PATH, X_SCALED_ALL_PATH]:
    print(required_path.name, "exists?", required_path.exists(), required_path)


PROCESSED_DIR       : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed
TRAINING_PLOTS_DIR : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\output\global_cancer\plots\training
LATENT_PLOTS_DIR   : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\output\global_cancer\plots\latent
MODELS_DIR         : C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\output\global_cancer\models\latent
X_train.npy exists? True C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\X_train.npy
X_val.npy exists? True C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\global_cancer\processed\X_val.npy
X_test.npy exists? True C:\Users\drziy\OneDrive\Desktop\Ali\scientific_research_current\global-cancer-complexity\data\g

In [3]:
X_train = np.load(X_TRAIN_PATH)
X_val   = np.load(X_VAL_PATH)
X_test  = np.load(X_TEST_PATH)

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)


Train: (194, 3000)
Val  : (42, 3000)
Test : (42, 3000)


In [4]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_tensor), batch_size=32)

In [5]:
class VAE(nn.Module):
    def __init__(self, input_dim=3000, latent_dim=10):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU()
        )
        
        self.mu_layer = nn.Linear(128, latent_dim)
        self.logvar_layer = nn.Linear(128, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.mu_layer(h), self.logvar_layer(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

In [6]:
def vae_loss(x, x_recon, mu, logvar, beta=0.01):
    recon_loss = nn.functional.mse_loss(x_recon, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

In [7]:
input_dim = X_train.shape[1]
latent_dim = 10

model = VAE(input_dim=input_dim, latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 50

train_losses = []
val_losses = []
train_recon_losses = []
train_kl_losses = []
val_recon_losses = []
val_kl_losses = []

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_recon = 0
    train_kl = 0
    
    for (x_batch,) in train_loader:
        x_batch = x_batch.to(device)
        
        optimizer.zero_grad()
        x_recon, mu, logvar = model(x_batch)
        loss, recon_loss, kl_loss = vae_loss(x_batch, x_recon, mu, logvar, beta=0.01)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_recon += recon_loss.item()
        train_kl += kl_loss.item()
    
    model.eval()
    val_loss = 0
    val_recon = 0
    val_kl = 0
    
    with torch.no_grad():
        for (x_batch,) in val_loader:
            x_batch = x_batch.to(device)
            x_recon, mu, logvar = model(x_batch)
            loss, recon_loss, kl_loss = vae_loss(x_batch, x_recon, mu, logvar, beta=0.01)
            val_loss += loss.item()
            val_recon += recon_loss.item()
            val_kl += kl_loss.item()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    train_kl_losses.append(train_kl)
    val_recon_losses.append(val_recon)
    val_kl_losses.append(val_kl)
    
    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Total: {train_loss:.2f} | Train Recon: {train_recon:.2f} | Train KL: {train_kl:.2f} | "
        f"Val Total: {val_loss:.2f} | Val Recon: {val_recon:.2f} | Val KL: {val_kl:.2f}"
    )

Epoch 001 | Train Total: 577629.79 | Train Recon: 577622.49 | Train KL: 729.58 | Val Total: 116218.59 | Val Recon: 116202.42 | Val KL: 1616.23
Epoch 002 | Train Total: 479084.85 | Train Recon: 478947.36 | Train KL: 13748.08 | Val Total: 94735.62 | Val Recon: 94695.82 | Val KL: 3980.39
Epoch 003 | Train Total: 403924.60 | Train Recon: 403709.50 | Train KL: 21509.53 | Val Total: 89366.19 | Val Recon: 89332.62 | Val KL: 3356.00
Epoch 004 | Train Total: 370716.33 | Train Recon: 370521.76 | Train KL: 19457.08 | Val Total: 83542.55 | Val Recon: 83496.68 | Val KL: 4587.73
Epoch 005 | Train Total: 345023.19 | Train Recon: 344814.23 | Train KL: 20896.04 | Val Total: 80081.41 | Val Recon: 80040.54 | Val KL: 4086.61
Epoch 006 | Train Total: 324604.81 | Train Recon: 324385.88 | Train KL: 21893.32 | Val Total: 79507.81 | Val Recon: 79464.83 | Val KL: 4298.35
Epoch 007 | Train Total: 314340.84 | Train Recon: 314111.26 | Train KL: 22957.52 | Val Total: 78392.63 | Val Recon: 78340.54 | Val KL: 5209.37

In [ ]:
epochs_axis = np.arange(1, epochs + 1)

# Standard total-loss training curve
plt.figure(figsize=(8, 5))
plt.plot(epochs_axis, train_losses, label="Train Total")
plt.plot(epochs_axis, val_losses, label="Validation Total")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("VAE Training Curve")
plt.tight_layout()
plt.savefig(TRAINING_PLOTS_DIR / "vae_training_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# More informative component diagnostic
plt.figure(figsize=(9, 6))
plt.plot(epochs_axis, train_recon_losses, label="Train Recon")
plt.plot(epochs_axis, val_recon_losses, label="Val Recon")
plt.plot(epochs_axis, train_kl_losses, label="Train KL")
plt.plot(epochs_axis, val_kl_losses, label="Val KL")
plt.xlabel("Epoch")
plt.ylabel("Loss Component")
plt.legend()
plt.title("VAE Loss Components")
plt.tight_layout()
plt.savefig(TRAINING_PLOTS_DIR / "vae_loss_components.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", TRAINING_PLOTS_DIR / "vae_training_curve.png")
print("Saved:", TRAINING_PLOTS_DIR / "vae_loss_components.png")


In [ ]:
X_all = np.load(X_SCALED_ALL_PATH)
X_all_tensor = torch.tensor(X_all, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    mu, logvar = model.encode(X_all_tensor)

latent = mu.cpu().numpy()

print("Latent shape:", latent.shape)


In [ ]:
# --- Save latent space (used by downstream notebooks) ---
np.save(LATENT_PATH, latent)

# --- Save model artifacts (reproducibility) ---
torch.save(model.state_dict(), MODEL_STATE_DICT_PATH)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "latent_dim": latent_dim,
    "input_dim": input_dim,
}, MODEL_CHECKPOINT_PATH)

# --- Confirm saves ---
print("Saved latent space to:", LATENT_PATH)
print("Saved model state_dict to:", MODEL_STATE_DICT_PATH)
print("Saved full checkpoint to:", MODEL_CHECKPOINT_PATH)


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(latent.ravel(), bins=50)
plt.xlabel("Latent value")
plt.ylabel("Frequency")
plt.title("Distribution of latent variables")
plt.tight_layout()
plt.savefig(LATENT_PLOTS_DIR / "latent_value_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", LATENT_PLOTS_DIR / "latent_value_histogram.png")
print("Latent mean:", float(latent.mean()))
print("Latent std :", float(latent.std()))
